In [1]:
import pandas as pd

# 请将 'wang1.csv' 替换为你实际文件的路径
file_path = '../../../5_Data/bupt_ring_selftest/wsx/wang2.csv'
parsed_rows = []

current_date, current_time, current_offset = "", "", ""

with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        
        # 忽略空行和 '}'
        if not line or line == '}':
            continue
            
        # 提取时间戳逻辑
        if ',' in line:
            parts = line.split(',', 3)
            current_date = parts[0].strip()
            current_time = parts[1].strip()
            current_offset = parts[2].strip()
            data_part = parts[3].strip()
        else:
            data_part = line
            
        row_dict = {
            'Date': current_date,
            'Time': current_time,
            'Offset': current_offset
        }
        
        # 提取各个传感器数据
        kv_pairs = data_part.split(';')
        for kv in kv_pairs:
            if ':' in kv:
                key, value = kv.split(':', 1)
                row_dict[key.strip()] = value.strip()
                
        parsed_rows.append(row_dict)

# 转换为表格格式
df = pd.DataFrame(parsed_rows)

# # 导出为整齐的 CSV 文件
# df.to_csv('cleaned_wang1.csv', index=False, encoding='utf-8-sig')
# print("数据处理完毕，已保存为 cleaned_wang1.csv")

In [2]:
df.columns = ['Date', 'Time', 'Duration', 'red', 'ied', 'accX', 'accY', 'accZ']
# 指定需要转换的列名
cols_to_convert = ['red', 'ied', 'accX', 'accY', 'accZ']

# 批量转换
df[cols_to_convert] = df[cols_to_convert].astype(float)

# 检查转换后的类型
print(df.dtypes)

Date            str
Time            str
Duration        str
red         float64
ied         float64
accX        float64
accY        float64
accZ        float64
dtype: object


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from ipywidgets import IntSlider, interact

In [4]:
df.dtypes

Date            str
Time            str
Duration        str
red         float64
ied         float64
accX        float64
accY        float64
accZ        float64
dtype: object

In [7]:
y = -1. * df['ied'].values
ymin, ymax = np.nanmin(y), np.nanmax(y)

@interact(
    start=IntSlider(min=0, max=len(y)-200, step=30, value=10000, description='start idx'),
    window=IntSlider(min=200, max=7900000, step=200, value=800, description='window')
)
def plot_zoom(start=0, window=500):
    end = min(start + window, len(y))
    t = np.arange(start, end) / 100.0
    plt.rcParams['font.size'] = 18
    fig, ax = plt.subplots(figsize=(18, 7))
    ax.plot(t, y[start:end], 'o-', linewidth=1, markersize=2)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('IED (high-pass)')
    # ax.set_ylim([ymin, ymax])

    ax.set_title(f'Sample {start}~{end} ({window} samples / {window/100:.1f}s)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=10000, description='start idx', max=1688335, step=30), IntSlider(value=8…

In [8]:
df.dtypes

Date            str
Time            str
Duration        str
red         float64
ied         float64
accX        float64
accY        float64
accZ        float64
dtype: object